# Lecture 19: Your Prompts + Final/Project Overview
I've collected a few of your submitted prompts to try out live in lecture using a slightly larger model! Hopefully this is a fun exercise that gets you thinking of novel LLM applications that you could apply to a project, either in this course or elsewhere.

I also want to take a moment to point out a few things to (hopefully) help you to write good prompt templates.
1. Think about what your prompt will look like once the replacement fields have been filled in with your input data. For example, if you write `"Think about the following {poem} and write an analysis..."`, the `{poem}` is just going show up smack dab in the middle of a sentence. You probably want to provide the input after you've written out some clear instructions. Always inspect a few filled-in templates to see what your prompt will actually look like to the model.
2. Prompts should be very specific when it comes to the desired output format. If you want a response to be limited to a certain format, length, style, or focus, you need to specify it and/or include several examples that exhibit those characteristics. 
3. With small models, I've found that it works better to only ask for one thing at a time, or to at least request concrete, few-token responses to a few carefully defined subquestions. For example, I would avoid prompts like `"Estimate x and y, accompanied by an analysis of z."` You might split each of these tasks into a separate, carefully constructed prompt instead of asking for all of them at once.
4. Write multiple prompts and test them. Every model behaves differently, and you might get better results by simply changing how you word your instructions. 

## Your prompts

### Setup

In [ ]:
import pandas as pd
import spacy
from openai import OpenAI
import json
from IPython.display import Markdown

In [ ]:
def get_response(prompt, model="qwen2.5:7b"):
    """Get a response from a chat model as a string given a prompt."""
    client = OpenAI(
        base_url="http://localhost:11434/v1", 
        api_key="ollama"
    )

    response = client.chat.completions.create(
        model=model, 
        messages = [
            {
                "role": "user", "content": prompt
            }
        ]
    )
    return response.choices[0].message.content

## Lemmatization and POS tagging 
Prompt by SI, also suggested by RR. 

### Why I chose this prompt
This is a tough task for a language model to do. The model receives input as subword tokens, so we're asking implicitly asking it to identify which token(s) correspond to each word, then to identify the lemma/POS for each of these words.

In [5]:
nlp = spacy.load("en_core_web_sm")

pos_tags = "\n".join([f"{label}: {spacy.explain(label)}" 
                      for label in spacy.parts_of_speech.NAMES.values()])

test_sentences = [
    "There's always a long line at Gimme Coffee after lecture.",
    "I was going to go get coffee but I had forgotten my wallet in the car."
]

lemmas_tags_spacy = [[(token.lemma_, token.pos_) for token in nlp(sentence)]
                     for sentence in test_sentences]

prompt = (
    "I want you to lemmatize the following English sentence. Output should be in the form of "
    "[(lemma, POS)...], so an array of tuples. Do not return anything else: just the list of tuples. "
    "Assign whichever POS tag (where POS means part of speech) that fits the context best, since "
    "some words can be in isolation polysemic. For POS, draw from the canonical POS categories, "
    "reiterated here: \n{pos_tags}\n\n"
    "Here is the sentence: \n{sentence}"
)

In [6]:
print(pos_tags)

: None
ADJ: adjective
ADP: adposition
ADV: adverb
AUX: auxiliary
CONJ: conjunction
CCONJ: coordinating conjunction
DET: determiner
INTJ: interjection
NOUN: noun
NUM: numeral
PART: particle
PRON: pronoun
PROPN: proper noun
PUNCT: punctuation
SCONJ: subordinating conjunction
SYM: symbol
VERB: verb
X: other
EOL: end of line
SPACE: space


In [7]:
lemmas_tags_spacy

[[('there', 'PRON'),
  ('be', 'VERB'),
  ('always', 'ADV'),
  ('a', 'DET'),
  ('long', 'ADJ'),
  ('line', 'NOUN'),
  ('at', 'ADP'),
  ('Gimme', 'PROPN'),
  ('Coffee', 'PROPN'),
  ('after', 'ADP'),
  ('lecture', 'NOUN'),
  ('.', 'PUNCT')],
 [('I', 'PRON'),
  ('be', 'AUX'),
  ('go', 'VERB'),
  ('to', 'PART'),
  ('go', 'VERB'),
  ('get', 'VERB'),
  ('coffee', 'NOUN'),
  ('but', 'CCONJ'),
  ('I', 'PRON'),
  ('have', 'AUX'),
  ('forget', 'VERB'),
  ('my', 'PRON'),
  ('wallet', 'NOUN'),
  ('in', 'ADP'),
  ('the', 'DET'),
  ('car', 'NOUN'),
  ('.', 'PUNCT')]]

In [8]:
for sentence in test_sentences: 
    response = get_response(
        prompt.format(pos_tags=pos_tags, sentence=sentence)
    )
    print("Sentence:\n", sentence, "\n")
    print("Response:\n", response, "\n")

Sentence:
 There's always a long line at Gimme Coffee after lecture. 

Response:
 [('There', 'PRON'), ("'", 'PUNCT'), ('s', 'PART'), ('always', 'ADV'), ('a', 'DET'), ('long', 'ADJ'), ('line', 'NOUN'), ('at', 'ADP'), ('Gimme', 'PROPN'), ('Coffee', 'PROPN'), ('after', 'ADP'), ('lecture', 'NOUN'), ('.', 'PUNCT')] 

Sentence:
 I was going to go get coffee but I had forgotten my wallet in the car. 

Response:
 [('I', 'PRON'), ('was', 'AUX'), ('going', 'VERB'), ('to', 'ADP'), ('go', 'VERB'), ('get', 'VERB'), ('coffee', 'NOUN'), ('but', 'CCONJ'), ('I', 'PRON'), ('had', 'AUX'), ('forgotten', 'VERB'), ('my', 'DET'), ('wallet', 'NOUN'), ('in', 'ADP'), ('the', 'DET'), ('car', 'NOUN'), ('.', 'PUNCT')] 



## Style transfer: haiku to rhyming poems
By AG.

### Why I chose this prompt
Style transfer is fun, but we are probably most familiar with this task in terms of images. We've all seen models that can transform a photo into a Renaissance painting or a comic book illustration (see this [space](https://huggingface.co/spaces/multimodalart/flux-style-shaping) for an example). This prompt attempts the same thing with poetic genres: given a poem in one style, we want to generate a semantically-equivalent poem in a different style. You could imagine doing the same thing for narratives, tweets, news articles, academic publications...

In [9]:
with open("basho_corpus.json") as f: 
    haikus = json.load(f)

haikus = [v.replace("/", "\n") for _, v in haikus.items()]
haikus = [haikus[0]] + [haikus[2]]
print(haikus[0], "\n")
print(haikus[1])

an old silent pond...
a frog jumps into the pond,
splash! Silence again. 

in the twilight rain
these brilliant-hued hibiscus --
a lovely sunset.


In [10]:
prompt = (
    "Take a 3 line 5-7-5 haiku and turn it into "
    "an 8 line free verse poem while still using "
    "the same subject matter. All eight lines should "
    "be different from the original, but about the same "
    "subject, and the last word at the end of each "
    "line should rhyme.\n"
    "The output should be a list of 8 text strings.\n"
    "{haiku}"
)

for haiku in haikus:
    response = get_response(
        prompt=prompt.format(haiku=haiku)
    )
    print("Original:\n", haiku, "\n")
    print("Generated:\n", response, "\n")

Original:
 an old silent pond...
a frog jumps into the pond,
splash! Silence again. 

Generated:
 mossy stones rest  
softly disturbed, water spreads—  
whispered secrets shared  
breath returning to stillness—  
lily pads lift, barely  
ripples fade like forgotten dreams—  
dawn breaks, light whispers—  
eternity's soft embrace ends 

Original:
 in the twilight rain
these brilliant-hued hibiscus --
a lovely sunset. 

Generated:
 1. evening's whispering rain —  
2. petals bloom in twilight’s glance —  
3. colors dance as droplets cling —  
4. hues melt into fading light —  
5. sky ignites with fiery crest —  
6. shadows stretch beneath the stars —  
7. night watches over nature's song —  
8. peace reigns until dawn's first gleam — 



## Recipe analysis
By TS

### Why I chose this prompt
This is a fun way to investigate what our model can infer given limited information. We give our model a list of ingredients from a recipe and ask for information about the cultural/historical factors that contributed to the creation of that recipe. I've written an alternate version of this prompt to demonstrate how adding some additional instructions can drastically change the model's output formatting. Because of how contemporary models are post-trained, we get a lot of extra filler that we probably don't want to sift through down the line. 

In [11]:
cookbooks = pd.read_csv("../data/cookbooks/feeding-america.csv")

cookbooks_slice = cookbooks.sample(2)
ingredients = (
    cookbooks_slice
    .ingredients
    .str.split(";")
    .str.join(", ")
    .tolist()
)

prompt = (
    "Read the following recipe from a historical cookbook: {ingredients}\n."
    "Analyze what this recipe reveals about the culture and daily life of its time."
)

modified_prompt = (
    "You are a culinary historian. Your job is to read a list of INGREDIENTS from a recipe found in a "
    "19th or 20th century cookbook and write a thorough ANALYSIS based on these INGREDIENTS. "
    "Your ANALYSIS should include insights into cultural factors that contributed to the creation of the recipe, as well "
    "as what these INGREDIENTS reveal about daily life when this recipe was written. "
    "Your ANALYSIS should be 3-5 sentences long.\n"
    "INGREDIENTS: {ingredients}\n"
    "ANALYSIS: "
)
cookbooks_slice[["ingredients", "ethnicgroup", "date"]]


,ingredients,ethnicgroup,date
40709,green pea;trout;water,NaN,1886
42465,brown sugar;cauliflower;celery seed;cucumber;f...,NaN,1909


In [13]:
for ing in ingredients:
    response_default = get_response(
        prompt.format(ingredients=ing)
    )
    response_modified = get_response(
        modified_prompt.format(ingredients=ing)
    )
    print("Ingredients: ", ing, "\n")
    print("Response (orig. prompt):\n", response_default, "\n")
    print("###Response (modified prompt):\n", response_modified, "\n***\n")

Ingredients:  green pea, trout, water 

Response (orig. prompt):
 The brief list of ingredients—from a historical cookbook—green peas, trout, and water provides valuable insights into the culture and daily life of its era. While it's somewhat incomplete (as such lists often lack detailed cooking steps or preparation methods), we can infer several points about the context in which this recipe likely originated:

1. **Accessibility to Fresh Produce**: The inclusion of fresh green peas suggests a regional or seasonal availability, indicating that the area had suitable agricultural conditions during certain times of the year for growing these crops. This would also point to a society where seasonal foods were highly valued and managed.

2. **Proximity to Water/Abundant Fishing**: Trout is a freshwater fish, implying proximity to bodies of fresh water such as rivers or lakes. The presence of water in the list might suggest that it was a necessary ingredient, possibly used for cooking purpos

## Project and Final Introduction

As described on the syllabus, you have two options for the final in this course. Both are assignments, to be completed at home, and they will be released following the PS4 due date (deadlines will communicated shortly). 

### Should I do the project or exam?
Grads are obligated to complete a final project independently, but undergrads have the choice of completing either the "exam" or the project in groups.
Logistically, the project and "exam" differ only in that *the project can be done in groups of up to three students*, whereas you *must complete the exam independently*. For the project, you should note that:
- every member of the group will receive the same grade, except in *extremely unusual circumstances*, and
- the work expected scales with the number of group members.

In practical terms, this means that a 3-person final project should be equivalent in quality and quantity to 3 individual projects or exams. This is not meant to scare you away from collaborating: in an individual project, you are performing *all* of the analysis and writing yourself, while a good group allows each participant to dedicate their time where they are most valuable. When forming a group, it will be helpful to identify in advance which parts of the project each group member will do, and when. 

In terms of the assignments' content, the key differences are:
- In the **exam**, you are supplied with a dataset and asked to perform a self-directed analysis
- In the **project**, you are asked to form a hypothesis and test it using some data you have sourced yourself. 
- In the **exam**, you will write everything (code and analyses) in a notebook.
- In the **project**, you will use a notebook for commented code, but write a short paper using a provided LaTeX template. 

### Project Advice
- Identify a good dataset as early as possible. Poke around in it and make sure there won't be any surprises once you get too deep into the rest of the project. Think about formatting, cleanliness, scope, and accessibility.
- Don't confuse a *research area* for a *research question*. "Gender and fiction" is a research area: it defines the intersection of two topics. "Are protagonists more likely to be female in post-1920 American fiction?" is a research question: you can perform an analysis to try and find an answer.
- If you are investigating a novel method (e.g. a prompt based approach), ensure that you have solid baselines against which to evaluate your method. 
- Give yourself/your team plenty of time to succeed. Do not leave this to the last minute. 
- The project is an opportunity for you to do real research: I want you to have something that you can show off on Github or even turn into a publishable paper. This is a great opportunity if you are thinking about grad school. 
